# Exogenous Data Sources — Ingestion

Fetches macro (NDX, DXY), Fear & Greed, Binance funding rate, Google Trends,
and Reddit sentiment data, then merges everything onto the BTC date index.

**Run this notebook before re-running `02_feature_analysis.ipynb`.**

Missing-history policy: pre-inception gaps stay `NaN` with a `<feature>_missing`
flag (never imputed with a neutral value). Calendar/resolution gaps
(weekends for NDX/DXY, weekly resolution for Google Trends) are forward-filled,
each with their own `_missing` flag marking interpolated days.

In [1]:
import yfinance as yf

# Nasdaq Composite = ^IXIC (NOT ^NDX, which is the Nasdaq-100)
try:
    macro_tickers = ["^IXIC", "DX-Y.NYB"]
    # Absolute start date (not a rolling "10y" window): BTC history starts
    # 2016-05-30, and a rolling window would both miss the first ~74 days on
    # every run and keep sliding forward, permanently losing more history.
    macro_data = yf.download(macro_tickers, start="2016-01-01", interval="1d")
    macro_data.to_csv("../data/raw/yahoo_macro.csv")
    print(macro_data.shape)
    print(macro_data.tail())
except Exception as e:
    print(f"WARNING: yahoo_macro fetch failed ({e}) - downstream merge will treat this source as unavailable")

/var/folders/dy/8mkjsrmx33v0zw8njf1n2bxr0000gn/T/ipykernel_23072/887661080.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  macro_data = yf.download(macro_tickers, start="2016-01-01", interval="1d")


[                       0%                       ]

[*********************100%***********************]  2 of 2 completed

(2670, 10)
Price            Close                      High                      Low  \
Ticker        DX-Y.NYB         ^IXIC    DX-Y.NYB         ^IXIC   DX-Y.NYB   
Date                                                                        
2026-08-06   99.970001  26348.349609  100.019997  26499.419922  99.639999   
2026-08-07   99.599998  26690.619141  100.000000  26712.619141  99.400002   
2026-08-10   99.809998  26605.359375   99.830002  26724.630859  99.580002   
2026-08-11   99.820000  26445.449219   99.900002  26679.259766  99.730003   
2026-08-12  100.000000  26588.488281  100.023003  26688.240234  99.612999   

Price                          Open                 Volume                
Ticker             ^IXIC   DX-Y.NYB         ^IXIC DX-Y.NYB         ^IXIC  
Date                                                                      
2026-08-06  26208.429688  99.660004  26268.839844      0.0  8.936850e+09  
2026-08-07  26478.009766  99.940002  26534.660156      0.0  8.183970e+09

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/yahoo_macro.csv", header=[0, 1], index_col=0)
df.index = pd.to_datetime(df.index)
assert df.shape[0] > 2000, f"expected >2000 rows, got {df.shape[0]}"
assert df.index.min().year <= 2017, f"expected data from ~2016, earliest is {df.index.min()}"
print("OK", df.shape, df.index.min(), df.index.max())

OK (2670, 10) 2016-01-04 00:00:00 2026-08-12 00:00:00


In [3]:
import requests
import pandas as pd

try:
    resp = requests.get("https://api.alternative.me/fng/?limit=0&format=json", timeout=30)
    resp.raise_for_status()
    fng_raw = resp.json()["data"]

    fng = pd.DataFrame(fng_raw)
    fng["Date"] = pd.to_datetime(fng["timestamp"].astype(int), unit="s").dt.normalize()
    fng["fear_greed_value"] = fng["value"].astype(float)
    fng = fng.rename(columns={"value_classification": "fear_greed_classification"})
    fng = fng[["Date", "fear_greed_value", "fear_greed_classification"]].sort_values("Date")
    fng = fng.set_index("Date")
    fng.to_csv("../data/raw/fear_greed.csv")
    print(fng.shape)
    print(fng.head())
    print(fng.tail())
except Exception as e:
    print(f"WARNING: fear_greed fetch failed ({e}) - downstream merge will treat this source as unavailable")

(3111, 2)
            fear_greed_value fear_greed_classification
Date                                                  
2018-02-01              30.0                      Fear
2018-02-02              15.0              Extreme Fear
2018-02-03              40.0                      Fear
2018-02-04              24.0              Extreme Fear
2018-02-05              11.0              Extreme Fear
            fear_greed_value fear_greed_classification
Date                                                  
2026-08-08              30.0                      Fear
2026-08-09              31.0                      Fear
2026-08-10              30.0                      Fear
2026-08-11              29.0                      Fear
2026-08-12              27.0                      Fear


In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/fear_greed.csv", index_col=0)
df.index = pd.to_datetime(df.index)
assert df["fear_greed_value"].between(0, 100).all(), "value out of [0,100] range"
assert df.index.min().year == 2018, f"expected earliest date in 2018, got {df.index.min()}"
print("OK", df.shape, df.index.min(), df.index.max())

OK (3111, 2) 2018-02-01 00:00:00 2026-08-12 00:00:00


In [5]:
import requests
import pandas as pd
import time

def fetch_binance_funding_rates(symbol="BTCUSDT", start_time_ms=1567900800000):
    """start_time_ms defaults to 2019-09-08, before BTCUSDT perpetual launch (2019-09-13)."""
    url = "https://fapi.binance.com/fapi/v1/fundingRate"
    rows = []
    start = start_time_ms
    while True:
        resp = requests.get(url, params={"symbol": symbol, "startTime": start, "limit": 1000}, timeout=30)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        rows.extend(batch)
        last_time = batch[-1]["fundingTime"]
        if last_time <= start:
            break
        start = last_time + 1
        if len(batch) < 1000:
            break
        time.sleep(0.3)
    return rows

try:
    funding_rows = fetch_binance_funding_rates()
    funding = pd.DataFrame(funding_rows)
    funding["Date"] = pd.to_datetime(funding["fundingTime"], unit="ms").dt.normalize()
    funding["fundingRate"] = funding["fundingRate"].astype(float)
    funding_daily = funding.groupby("Date")["fundingRate"].mean().rename("funding_rate").to_frame()
    funding_daily.to_csv("../data/raw/funding_rates.csv")
    print(funding_daily.shape)
    print(funding_daily.head())
    print(funding_daily.tail())
except Exception as e:
    print(f"WARNING: funding_rates fetch failed ({e}) - downstream merge will treat this source as unavailable")

(2529, 1)
            funding_rate
Date                    
2019-09-10        0.0001
2019-09-11        0.0001
2019-09-12        0.0001
2019-09-13        0.0001
2019-09-14        0.0001
            funding_rate
Date                    
2026-08-08      0.000056
2026-08-09      0.000060
2026-08-10      0.000068
2026-08-11      0.000042
2026-08-12      0.000084


In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/funding_rates.csv", index_col=0)
df.index = pd.to_datetime(df.index)
assert df.index.min().year == 2019, f"expected earliest date in 2019, got {df.index.min()}"
assert df["funding_rate"].abs().max() < 1.0, "funding rate magnitude looks wrong (should be a small fraction)"
print("OK", df.shape, df.index.min(), df.index.max())

OK (2529, 1) 2019-09-10 00:00:00 2026-08-12 00:00:00


In [7]:
from pytrends.request import TrendReq
import pandas as pd
import time

def fetch_google_trends(keyword="Bitcoin", timeframe="2016-01-01 2026-08-12", retries=3):
    pytrends = TrendReq(hl="en-US", tz=0)
    for attempt in range(retries):
        try:
            pytrends.build_payload([keyword], timeframe=timeframe)
            return pytrends.interest_over_time()
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"Google Trends request failed ({e}), retrying in 10s...")
            time.sleep(10)

try:
    trends = fetch_google_trends()
    trends = trends.drop(columns=["isPartial"], errors="ignore")
    trends = trends.rename(columns={"Bitcoin": "google_trends_score"})
    trends.index.name = "Date"
    # Look-ahead bias fix: pytrends returns one row per calendar month, dated
    # to the 1st of that month, but the value is a whole-month aggregate that
    # isn't actually known until the month ends. Relabel each row to the 1st
    # of the FOLLOWING month so e.g. March's aggregate attaches to 2020-04-01
    # instead of 2020-03-01, where it would leak the rest of March forward.
    trends.index = trends.index + pd.offsets.MonthBegin(1)
    trends.to_csv("../data/raw/google_trends.csv")
    print(trends.shape)
    print(trends.head())
    print(trends.tail())
except Exception as e:
    print(f"WARNING: google_trends fetch failed after retries ({e}) - downstream merge will treat this source as unavailable")

(128, 1)
            google_trends_score
Date                           
2016-02-01                    3
2016-03-01                    3
2016-04-01                    3
2016-05-01                    3
2016-06-01                    4
            google_trends_score
Date                           
2026-05-01                   25
2026-06-01                   23
2026-07-01                   27
2026-08-01                   21
2026-09-01                   16


/Users/vivienbistrel/AI4Finance/.venv/lib/python3.13/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


In [8]:
import pandas as pd

df = pd.read_csv("../data/raw/google_trends.csv", index_col=0)
df.index = pd.to_datetime(df.index)
assert df["google_trends_score"].between(0, 100).all(), "score out of [0,100] range"
assert df.index.min().year <= 2017, f"expected data from ~2016, earliest is {df.index.min()}"
# Look-ahead bias fix check: dates are shifted forward one month from the raw
# fetch's month-start labels (see MonthBegin(1) shift above), so the earliest
# date must be strictly AFTER the fetch timeframe's own start (2016-01-01).
assert df.index.min() > pd.Timestamp("2016-01-01"), (
    f"expected dates shifted past the raw fetch start, got earliest {df.index.min()} "
    "-- the MonthBegin(1) shift may not have been applied"
)
print("OK", df.shape, df.index.min(), df.index.max())

OK (128, 1) 2016-02-01 00:00:00 2026-09-01 00:00:00


In [9]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Arctic Shift republishes the old Pushshift Reddit archive: free, no auth,
# no key. Reddit's own API is effectively closed to new developers (2026
# Responsible Builder Policy) and X/Twitter has no purchasable historical
# access at any price for a new developer — see the design spec for the
# full investigation.
#
# Arctic Shift returns results newest-first by default, and r/Bitcoin gets
# more than 100 posts/day on the vast majority of days in this window — a
# single limit=100 request per day would silently truncate the early hours
# of almost every high-volume day, which is a fabricated-data violation
# (missing early buckets would be recorded as volume=0 instead of "not
# measured"). Instead we page through ALL of a day's posts using ascending
# order and a moving cursor, mirroring the cursor-pagination pattern used
# above for Binance funding rates (fetch_binance_funding_rates). This makes
# most days need 2+ pages instead of 1, so expect this cell to take roughly
# 40-90+ minutes (up from ~15-25) — it is now doing an exhaustive fetch
# rather than a bounded sample. That is expected, not a hang.

FETCH_START = pd.Timestamp("2020-01-01")
FETCH_END = pd.Timestamp("2026-05-26")
full_range = pd.date_range(FETCH_START, FETCH_END, freq="D")

analyzer = SentimentIntensityAnalyzer()
records = []
failed_days = set()


def fetch_day_posts(subreddit, day_start, day_end, page_limit=100):
    """Paginate through ALL posts in [day_start, day_end) using ascending
    order and a moving cursor, so no posts are silently dropped on
    high-volume days (the API returns newest-first by default, which
    would otherwise systematically truncate the early hours of every
    high-volume day)."""
    posts = []
    cursor_after = day_start
    while True:
        resp = requests.get(
            "https://arctic-shift.photon-reddit.com/api/posts/search",
            params={
                "subreddit": subreddit,
                "after": cursor_after,
                "before": day_end,
                "limit": page_limit,
                "sort": "asc",
            },
            timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json().get("data", [])
        if not batch:
            break
        posts.extend(batch)
        last_created = max(p["created_utc"] for p in batch)
        next_cursor = datetime.fromtimestamp(last_created + 1, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        if len(batch) < page_limit:
            break
        if next_cursor == cursor_after:
            break  # safety valve against an infinite loop if the cursor can't advance
        cursor_after = next_cursor
    return posts


day = FETCH_START
while day <= FETCH_END:
    day_start = day.strftime("%Y-%m-%d")
    day_end = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    day_posts = None
    for attempt in range(3):
        try:
            day_posts = fetch_day_posts("Bitcoin", day_start, day_end)
            break
        except Exception as e:
            if attempt < 2:
                print(f"WARNING: Arctic Shift fetch failed for {day_start} (attempt {attempt + 1}/3, {e}), retrying...")
                time.sleep(1 * (attempt + 1))
            else:
                print(f"WARNING: Arctic Shift fetch failed for {day_start} after 3 attempts ({e}), marking as missing")
                failed_days.add(pd.Timestamp(day.date()))
    if day_posts:
        for p in day_posts:
            text = f"{p.get('title', '')} {p.get('selftext') or ''}"
            score = analyzer.polarity_scores(text)["compound"]
            dt = datetime.fromtimestamp(p["created_utc"], tz=timezone.utc)
            bucket = (dt.hour // 4) * 4
            records.append({"Date": dt.date(), "bucket": bucket, "polarity": score})
    day += pd.Timedelta(days=1)
    time.sleep(0.15)

print(f"Collected {len(records)} posts across the fetch window")
print(f"Failed days: {len(failed_days)} / {len(full_range)}")

posts_df = pd.DataFrame(records)
if not posts_df.empty:
    posts_df["Date"] = pd.to_datetime(posts_df["Date"])

wide = pd.DataFrame(index=full_range)
wide.index.name = "Date"

failed_index = pd.DatetimeIndex(sorted(failed_days))

for h in (0, 4, 8, 12, 16, 20):
    if posts_df.empty:
        volume = pd.Series(0, index=full_range, dtype="float64")
        polarity = pd.Series(pd.NA, index=full_range, dtype="float64")
    else:
        bucket_df = posts_df[posts_df["bucket"] == h]
        daily = bucket_df.groupby("Date")["polarity"].agg(["mean", "count"])
        daily = daily.reindex(full_range)
        volume = daily["count"].fillna(0).astype("float64")
        polarity = daily["mean"].where(volume > 0)
    # Days where the fetch itself failed after retries have no real
    # measurement at all (not a genuine "no posts" reading) — mark every
    # bucket NaN so the merge cell's *_missing logic flags them correctly,
    # instead of fabricating a zero.
    volume.loc[volume.index.isin(failed_index)] = pd.NA
    polarity.loc[polarity.index.isin(failed_index)] = pd.NA
    wide[f"reddit_polarity_{h}h"] = polarity
    wide[f"reddit_volume_{h}h"] = volume

wide.to_csv("../data/raw/reddit_sentiment.csv")
print(wide.shape)
print(wide.head())
print(wide.tail())

Collected 569755 posts across the fetch window
Failed days: 0 / 2338


(2338, 12)
            reddit_polarity_0h  reddit_volume_0h  reddit_polarity_4h  \
Date                                                                   
2020-01-01            0.260764              14.0            0.042635   
2020-01-02            0.160552              27.0            0.183400   
2020-01-03            0.197865              26.0            0.033650   
2020-01-04            0.092568              22.0            0.157538   
2020-01-05            0.241700              32.0            0.274641   

            reddit_volume_4h  reddit_polarity_8h  reddit_volume_8h  \
Date                                                                 
2020-01-01              17.0            0.413554              13.0   
2020-01-02               7.0            0.137085              26.0   
2020-01-03              22.0            0.073841              29.0   
2020-01-04              29.0            0.056585              13.0   
2020-01-05              22.0            0.204700              28

In [10]:
import pandas as pd

df = pd.read_csv("../data/raw/reddit_sentiment.csv", index_col=0)
df.index = pd.to_datetime(df.index)

expected_cols = [f"reddit_{stat}_{h}h" for h in (0, 4, 8, 12, 16, 20) for stat in ("polarity", "volume")]
missing_cols = [c for c in expected_cols if c not in df.columns]
assert not missing_cols, f"missing columns: {missing_cols}"

# Volume is NaN exactly on days where the fetch failed after retries
# (recorded as missing, not a fabricated zero) — so the invariant now
# covers three states per bucket: volume>0 (real activity, polarity
# present), volume==0 (real "no posts" reading, polarity NaN), and
# volume NaN (fetch failed, polarity NaN too).
for h in (0, 4, 8, 12, 16, 20):
    vol = df[f"reddit_volume_{h}h"]
    pol = df[f"reddit_polarity_{h}h"]
    assert (vol.dropna() >= 0).all(), f"negative volume found in reddit_volume_{h}h"
    assert (pol.isna() == ~(vol > 0)).all(), f"polarity/volume mismatch in reddit_polarity_{h}h — polarity must be NaN exactly where volume is 0 or missing"
    assert pol.dropna().between(-1, 1).all(), f"polarity out of [-1,1] range in reddit_polarity_{h}h"

assert df.index.min() == pd.Timestamp("2020-01-01"), f"expected start 2020-01-01, got {df.index.min()}"
assert df.index.max() == pd.Timestamp("2026-05-26"), f"expected end 2026-05-26, got {df.index.max()}"

# Coverage check: with a correct exhaustive-pagination fetch of an active
# subreddit, a day with truly zero posts across all 6 buckets (whether a
# genuine all-day silence or a failed fetch marked missing) should be rare.
# A large count here signals a systematic failure (e.g. a total API outage)
# rather than normal daily fluctuation.
volume_cols = [f"reddit_volume_{h}h" for h in (0, 4, 8, 12, 16, 20)]
total_volume_per_day = df[volume_cols].sum(axis=1)
days_with_zero_total = int((total_volume_per_day == 0).sum())
assert days_with_zero_total < 100, f"{days_with_zero_total} days have zero total Reddit volume across all buckets — check the fetch (likely a total outage or systematic failure), not a normal daily fluctuation"
print(f"Days with zero total Reddit volume: {days_with_zero_total} / {len(df)}")

print("OK", df.shape)

Days with zero total Reddit volume: 0 / 2338
OK (2338, 12)


In [11]:
import os
import pandas as pd

btc = pd.read_csv("../data/processed/yahooFinanceDataCleaned.csv", index_col=0)
btc.index = pd.to_datetime(btc.index)

# Guard against silently producing an all-NaN merged output if the tracked
# BTC CSV is ever wrong (wrong shape, wrong index type) — this notebook is
# meant to be re-run unattended, so a bad input must raise loudly, not merge
# into a wall of NaNs with no error.
assert len(btc) > 3000, f"BTC data looks too short ({len(btc)} rows) — check data/processed/yahooFinanceDataCleaned.csv"
assert btc.index.min().year <= 2017, f"BTC data doesn't start near 2016 (starts {btc.index.min()}) — check the Date index parsed correctly"

date_index = btc.index

exogenous = pd.DataFrame(index=date_index)
exogenous.index.name = "Date"


def load_raw_csv(path, header=0):
    if not os.path.exists(path):
        print(f"WARNING: {path} not found, its columns will be entirely NaN/missing")
        return None
    df = pd.read_csv(path, index_col=0, header=header)
    df.index = pd.to_datetime(df.index)
    return df


# --- Macro (NDX, DXY): market-closed gaps get forward-filled ---
macro_raw = load_raw_csv("../data/raw/yahoo_macro.csv", header=[0, 1])
if macro_raw is not None:
    macro = pd.DataFrame(index=macro_raw.index)
    macro["NDX_Close"] = macro_raw["Close"]["^IXIC"]
    macro["DXY_Close"] = macro_raw["Close"]["DX-Y.NYB"]
    macro = macro.reindex(date_index)
else:
    macro = pd.DataFrame({"NDX_Close": pd.NA, "DXY_Close": pd.NA}, index=date_index)

for col in ["NDX_Close", "DXY_Close"]:
    exogenous[f"{col}_missing"] = macro[col].isna().astype(int)
    exogenous[col] = macro[col].ffill()

# --- Fear & Greed: genuine pre-2018 non-existence, NaN stays NaN ---
fng = load_raw_csv("../data/raw/fear_greed.csv")
fng = fng.reindex(date_index) if fng is not None else pd.DataFrame({"fear_greed_value": pd.NA}, index=date_index)
exogenous["fear_greed_value"] = fng["fear_greed_value"]
exogenous["fear_greed_value_missing"] = fng["fear_greed_value"].isna().astype(int)

# --- Funding rate: genuine pre-2019-09 non-existence, NaN stays NaN ---
funding = load_raw_csv("../data/raw/funding_rates.csv")
funding = funding.reindex(date_index) if funding is not None else pd.DataFrame({"funding_rate": pd.NA}, index=date_index)
exogenous["funding_rate"] = funding["funding_rate"]
exogenous["funding_rate_missing"] = funding["funding_rate"].isna().astype(int)

# --- Google Trends: native since 2016 but weekly resolution, forward-filled ---
trends = load_raw_csv("../data/raw/google_trends.csv")
trends = trends.reindex(date_index) if trends is not None else pd.DataFrame({"google_trends_score": pd.NA}, index=date_index)
exogenous["google_trends_score_missing"] = trends["google_trends_score"].isna().astype(int)
exogenous["google_trends_score"] = trends["google_trends_score"].ffill()

# --- Reddit: 4h-bucketed sentiment; only its fetch window (2020-01-01 to
# 2026-05-26) has data. volume=0 is a real "no posts that bucket" reading
# (not missing); polarity is NaN exactly where volume is 0 (undefined, not
# imputed as neutral) — both enforced upstream in the fetch cell already,
# this block just applies the same reindex-and-flag pattern as every other
# source above. reindex is given explicit columns=reddit_cols (rather than
# just index=date_index) so that a stale/old-schema reddit_sentiment.csv
# (e.g. left over from before the 12-column bucket format) degrades to
# full-NaN columns instead of raising a KeyError below.
reddit_cols = [f"reddit_{stat}_{h}h" for h in (0, 4, 8, 12, 16, 20) for stat in ("polarity", "volume")]
reddit_daily = load_raw_csv("../data/raw/reddit_sentiment.csv")
if reddit_daily is not None:
    reddit_daily = reddit_daily.reindex(index=date_index, columns=reddit_cols)
else:
    reddit_daily = pd.DataFrame({c: pd.NA for c in reddit_cols}, index=date_index)
for col in reddit_cols:
    exogenous[f"{col}_missing"] = reddit_daily[col].isna().astype(int)
    exogenous[col] = reddit_daily[col]

exogenous.to_csv("../data/raw/exogenous_merged.csv")
print(exogenous.shape)
print(exogenous.isna().mean().rename("pct_nan"))

(3653, 34)
NDX_Close_missing              0.000000
NDX_Close                      0.000274
DXY_Close_missing              0.000000
DXY_Close                      0.000274
fear_greed_value               0.168629
fear_greed_value_missing       0.000000
funding_rate                   0.327950
funding_rate_missing           0.000000
google_trends_score_missing    0.000000
google_trends_score            0.000547
reddit_polarity_0h_missing     0.000000
reddit_polarity_0h             0.361073
reddit_volume_0h_missing       0.000000
reddit_volume_0h               0.359978
reddit_polarity_4h_missing     0.000000
reddit_polarity_4h             0.360526
reddit_volume_4h_missing       0.000000
reddit_volume_4h               0.359978
reddit_polarity_8h_missing     0.000000
reddit_polarity_8h             0.360526
reddit_volume_8h_missing       0.000000
reddit_volume_8h               0.359978
reddit_polarity_12h_missing    0.000000
reddit_polarity_12h            0.360252
reddit_volume_12h_missing    

In [12]:
import pandas as pd

df = pd.read_csv("../data/raw/exogenous_merged.csv", index_col=0)
df.index = pd.to_datetime(df.index)

expected_cols = [
    "NDX_Close", "NDX_Close_missing", "DXY_Close", "DXY_Close_missing",
    "fear_greed_value", "fear_greed_value_missing",
    "funding_rate", "funding_rate_missing",
    "google_trends_score", "google_trends_score_missing",
] + [f"reddit_{stat}_{h}h{suffix}" for h in (0, 4, 8, 12, 16, 20) for stat in ("polarity", "volume") for suffix in ("", "_missing")]
missing_cols = [c for c in expected_cols if c not in df.columns]
assert not missing_cols, f"missing columns: {missing_cols}"

# Pre-inception rows must be flagged missing, not filled. Boundaries are
# EXCLUSIVE of the genuine first-data day (2018-02-01 for Fear & Greed,
# 2019-09-10 for funding rate) -- the original brief's inclusive slices
# asserted real data was missing and always failed against live data
# (see Task 7 report); fixed here to express the actual intent.
pre_2018 = df.loc[:"2018-01-31"]
assert (pre_2018["fear_greed_value_missing"] == 1).all(), "fear_greed should be missing before Feb 2018"
assert pre_2018["fear_greed_value"].isna().all(), "fear_greed should be NaN (not imputed) before Feb 2018"

pre_2019_09 = df.loc[:"2019-09-09"]
assert (pre_2019_09["funding_rate_missing"] == 1).all(), "funding_rate should be missing before Sept 2019"
assert pre_2019_09["funding_rate"].isna().all(), "funding_rate should be NaN (not imputed) before Sept 2019"

print("OK", df.shape)

OK (3653, 34)
